Train Weekly Model - Overall, Live Production Data
Trains LightGBM on a units-per-working-day rate, then scales predictions by the actual number of working days (accounting for Sundays and public holidays) in the target week.

In [0]:
dbutils.library.restartPython()

In [0]:
%run ../../_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_gold
import pandas as pd
import mlflow
import mlflow.lightgbm
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

blob_service = get_blob_service(storage_account_name, storage_account_key)

def wape(y_true, y_pred):
    return abs(y_true - y_pred).sum() / abs(y_true).sum()

def compute_working_days(period_start, period_end, holiday_dates_set):
    all_days = pd.date_range(period_start, period_end)
    return sum(1 for d in all_days if d.weekday() != 6 and d.date() not in holiday_dates_set)

Load gold and holiday calendar

In [0]:
gold_weekly = read_gold(blob_service, "live/battery/data/phase1_overall_weekly_live.parquet")
gold_weekly["week_start"] = pd.to_datetime(gold_weekly["week_start"])

holiday_calendar = read_gold(blob_service, "live/battery/data/reference/holiday_calendar.parquet")
holiday_calendar["date"] = pd.to_datetime(holiday_calendar["date"])
holiday_dates_set = set(holiday_calendar["date"].dt.date)

print(gold_weekly[["week_start", "working_days", "units_per_working_day", "rate_lag_4w", "rate_rolling_avg_4w"]].tail(10))

Prepare rate-based features, train/test split

In [0]:
feature_cols_rate = ["week_of_year", "month", "contains_month_end", "rate_lag_4w", "rate_rolling_avg_4w"]
target_col_rate = "units_per_working_day"

model_data_rate = gold_weekly.dropna(subset=feature_cols_rate + [target_col_rate]).copy()
model_data_rate = model_data_rate.sort_values("week_start")

split_idx = int(len(model_data_rate) * 0.8)
train_rate = model_data_rate.iloc[:split_idx]
test_rate = model_data_rate.iloc[split_idx:]

print(f"Train: {len(train_rate)} weeks, Test: {len(test_rate)} weeks")

X_train_r, y_train_r = train_rate[feature_cols_rate], train_rate[target_col_rate]
X_test_r, y_test_r = test_rate[feature_cols_rate], test_rate[target_col_rate]

Train, evaluate, log to MLflow

In [0]:
with mlflow.start_run(run_name="phase1_overall_weekly_rate_based"):
    params = {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 6, "random_state": 42}
    mlflow.log_params(params)

    model_rate = lgb.LGBMRegressor(**params)
    model_rate.fit(X_train_r, y_train_r)

    rate_preds = model_rate.predict(X_test_r)
    total_preds = rate_preds * test_rate["working_days"].values
    actual_totals = test_rate["total_units_sold"].values

    mae = mean_absolute_error(actual_totals, total_preds)
    rmse = mean_squared_error(actual_totals, total_preds) ** 0.5
    wape_score = wape(actual_totals, total_preds)

    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("wape", wape_score)
    mlflow.lightgbm.log_model(model_rate, name="model", input_example=X_train_r.head(3))

    print(f"Rate-based Model — MAE: {mae:.2f}   RMSE: {rmse:.2f}   WAPE: {wape_score:.3%}")

    baseline_totals = test_rate["rate_rolling_avg_4w"].values * test_rate["working_days"].values
    baseline_mae = mean_absolute_error(actual_totals, baseline_totals)
    baseline_wape = wape(actual_totals, baseline_totals)
    print(f"Rate-based Baseline — MAE: {baseline_mae:.2f}   WAPE: {baseline_wape:.3%}")

Feature importance check

In [0]:
importance = pd.DataFrame({
    "feature": feature_cols_rate,
    "importance": model_rate.feature_importances_
}).sort_values("importance", ascending=False)
print(importance)

Retrain on all data, predict next week

In [0]:
X_all_r = model_data_rate[feature_cols_rate]
y_all_r = model_data_rate[target_col_rate]

final_model_rate = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42)
final_model_rate.fit(X_all_r, y_all_r)

last_week_start = gold_weekly["week_start"].max()
next_week_start = last_week_start + pd.Timedelta(weeks=1)
next_week_end = next_week_start + pd.Timedelta(days=6)

next_week_working_days = compute_working_days(next_week_start, next_week_end, holiday_dates_set)
next_week_working_days = max(next_week_working_days, 1)

lag_row = gold_weekly[gold_weekly["week_start"] == next_week_start - pd.Timedelta(weeks=4)]
rate_lag_value = lag_row["units_per_working_day"].values[0] if len(lag_row) > 0 else None

next_week_features = pd.DataFrame([{
    "week_of_year": next_week_start.isocalendar()[1],
    "month": next_week_start.month,
    "contains_month_end": int(next_week_start.month != next_week_end.month),
    "rate_lag_4w": rate_lag_value,
    "rate_rolling_avg_4w": gold_weekly["units_per_working_day"].tail(4).mean(),
}])

predicted_rate = final_model_rate.predict(next_week_features[feature_cols_rate])[0]
predicted_total = predicted_rate * next_week_working_days

print(f"Week of {next_week_start.date()}: {next_week_working_days} working days (out of 6 possible)")
print(f"Predicted rate: {predicted_rate:.1f} units/working day")
print(f"Predicted total: {predicted_total:.0f} units")